# Modelo CART — Análisis de Cartera
**Datos:** Diciembre 2025

In [45]:
import pandas as pd
import numpy as np
import os

RUTA = r'C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\Gadiel\Aplicacion\OneDrive_1_21-3-2026'

# Archivos disponibles
print('Archivos en la carpeta:')
for f in os.listdir(RUTA):
    print(' -', f)

Archivos en la carpeta:
 - 1022416832_WRH91L_boardingPass.pdf
 - 20250131_anexo_1_cap_2_tit_4.pdf
 - 20250624_circ_contable_finan_completa.pdf
 - CalCartera_Analisis_Utrah._20251211_centrales.xlsx
 - Criterios de evaluación.docx
 - Detalle tabla evaluación de cartera curuba.xlsx
 - Informe_Cartera_9744.xlsx
 - SALIDA CALIF CARTERA ENT_ DIC_2025_centrales_.xlsx
 - SALIDA PEC SALDOS_DIC_2025_centrales_.xlsx


## Categorías de riesgo y altura de mora por modalidad

| Categoría | Vivienda | Consumo | Microcrédito | Comercial |
|---|---|---|---|---|
| A — Normal | ≤ 60 días | ≤ 30 días | ≤ 30 días | ≤ 30 días |
| B — Aceptable | 61–150 días | 31–60 días | 31–60 días | 31–90 días |
| C — Apreciable | 151–360 días | 61–90 días | 61–90 días | 91–120 días |
| D — Significativo | 361–540 días | 91–180 días | 91–120 días | 121–150 días |
| E — Irrecuperable | > 540 días | > 180 días | > 120 días | > 150 días |

## Consideraciones clave para el modelo

1. **Variable objetivo (calificación):** el modelo debe predecir la categoría de riesgo A–E,
que se determina principalmente por la altura de mora, pero siempre tomando la
calificación de mayor riesgo entre todas las fuentes posibles.

2. **Deterioro como variable de salida:** si el modelo también estima provisiones, debes
parametrizar los porcentajes de deterioro diferenciados por modalidad (consumo,
comercial persona natural/jurídica, vivienda, microcrédito), incluyendo las subcategorías
E1 y E2.

3. **Variables relevantes para el modelo Logit:** altura de mora, modalidad de crédito,
tipo de persona (natural/jurídica), existencia y tipo de garantía (hipotecaria vs. no
hipotecaria), valor de aportes sociales, y condición de reestructuración.

4. **Regla del arrastre:** debes considerar que la calificación no es solo individual — si
un deudor tiene múltiples créditos de la misma modalidad, todos heredan la peor
calificación, lo que afecta la definición del incumplimiento en tu set de entrenamiento.

In [46]:
os.chdir("C:/Users/Usuario/OneDrive - Global Green Growth Institute/Documentos/Gadiel/Aplicacion/OneDrive_1_21-3-2026")

In [47]:
data=pd.read_excel("Informe_Cartera_9744.xlsx",header=3)
data.head(3)

,VIGENCIA,MES,DÍA,CTVO,TIPO IDENTIFICACIÓN,IDETIFICACIÓN,CÓDIGO CONTABLE,MODIFICACIONES AL CRÉDITO,CRÉDITO,CODCUE,...,USUARIO ANALISTA,FECHA INICIO,FLUJO ESPECIAL,INGRESOS ACTUALES,INGRESOS AÑO ANTERIOR,ÚLTIMA ACTUALIZACIÓN DE DATOS,ACTIVOS,PASIVOS,SECTOR ECONOMICO,FECHA SIGUIENTE CUOTA
0,2026,1,31,1,C,957030829,144205,4,157011115493,11115493,...,ND,2012-10-09,NO,7612000,76120000.0,2025-06-14,82779000.0,30000000.0,SERVICIOS Y OTRAS ACTIVIDADES,2026-01-31
1,2026,1,31,2,C,3870877128,144220,3,301111410117,11410117,...,OLGALUCIA,2021-05-06,NO,1884596,18845960.0,2021-04-29,14232000.0,8500000.0,SERVICIOS Y OTRAS ACTIVIDADES,2025-10-18
2,2026,1,31,3,C,1743121542,144205,3,2601111448998,11448998,...,DANIELA,2022-05-26,NO,3600000,36000000.0,2022-10-12,20000000.0,4342000.0,SERVICIOS Y OTRAS ACTIVIDADES,2026-02-15


In [48]:
data.columns

Index(['VIGENCIA', 'MES', 'DÍA', 'CTVO', 'TIPO IDENTIFICACIÓN',
       'IDETIFICACIÓN', 'CÓDIGO CONTABLE', 'MODIFICACIONES AL CRÉDITO',
       'CRÉDITO', 'CODCUE', 'FECHA DESEMBOLSO', 'FECHA VENCIMIENTO',
       'MOROSIDAD', 'TIPO CUOTA', 'CUOTAS PAGADAS', 'AMORTIZACIÓN',
       'MODALIDAD', 'TASA INTERÉS NOMINAL', 'TASA INTERÉS EFECTIVA',
       'VALOR PRÉSTAMO', 'VALOR CUOTA', 'SALDO CAPITAL', 'SALDO INTERESES',
       'OTROS SALDOS', 'GARANTÍA', 'FECHA AVALÚO', 'DETERIORO',
       'DETERIORO INTERESES', 'CONTINGENCIA', 'VALOR CUOTA EXTRA',
       'MESES CUOTA EXTRA', 'FECHA ÚLTIMO PAGO', 'CLASE GARANTÍA',
       'DESTINO CRÉDITO', 'CÓDIGO OFICINA', 'PERIODICIDAD AMORTIZACIÓN',
       'VALOR MORA', 'TIPO VIVIENDA', 'SEÑAL VIS ', 'TIPO O RANGO VIVIENDA',
       'SEÑAL SUBSIDIO', 'ENTIDAD REDESCUENTO', 'MARGEN REDESCUENTO',
       'DESEMBOLSO', 'MONEDA', 'FECHA MODIFICACIÓN',
       'CALIFICACIÓN ANTES MODIFICACIÓN', 'VALOR APORTES SOCIALES',
       'VALOR LÍNEA CRÉDITO', 'MODIFICACION

## 1. Calificación de Cartera en centrales (54,384 clientes)

In [42]:
df_calif = pd.read_excel(
    os.path.join(RUTA, 'SALIDA CALIF CARTERA ENT_ DIC_2025_centrales_.xlsx'),
    sheet_name='Calificacion_Cartera'
)

print('Shape:', df_calif.shape)
print('\nColumnas:')
print(df_calif.columns.tolist())
df_calif.head(3)

Shape: (54384, 23)

Columnas:
['AGENCIA', 'TIPO_ID', 'NUMERO_ID', 'NOMBRE', 'SALDO_TOTAL', 'SALDO_ENTIDAD', 'SALDO A OCT', 'Columna1', 'SALDO MORA', 'PARTICIPACIÓN_SALDO_ENTIDAD', 'PART_CALIF_A', 'PART_CALIF_B', 'PART_CALIF_C', 'PART_CALIF_D', 'PART_CALIF_E', 'CALIF_DE_ARRASTRE_SUPER_MAYOR_20%', 'CALIF_DE_ARRASTRE_ACIDA_MAYOR_20%', 'PROVISION_CON_CALIFICACION_SUPER_MAYOR_20%', 'PROVISION_CON_CALIFICACION_ACIDA_MAYOR_20%', 'CALIF_DE_ARRASTRE_SUPER_MAYOR_25%', 'CALIF_DE_ARRASTRE_ACIDA_MAYOR_25%', 'PROVISION_CON_CALIFICACION_SUPER_MAYOR_25%', 'PROVISION_CON_CALIFICACION_ACIDA_MAYOR_25%']


,AGENCIA,TIPO_ID,NUMERO_ID,NOMBRE,SALDO_TOTAL,SALDO_ENTIDAD,SALDO A OCT,Columna1,SALDO MORA,PARTICIPACIÓN_SALDO_ENTIDAD,...,PART_CALIF_D,PART_CALIF_E,CALIF_DE_ARRASTRE_SUPER_MAYOR_20%,CALIF_DE_ARRASTRE_ACIDA_MAYOR_20%,PROVISION_CON_CALIFICACION_SUPER_MAYOR_20%,PROVISION_CON_CALIFICACION_ACIDA_MAYOR_20%,CALIF_DE_ARRASTRE_SUPER_MAYOR_25%,CALIF_DE_ARRASTRE_ACIDA_MAYOR_25%,PROVISION_CON_CALIFICACION_SUPER_MAYOR_25%,PROVISION_CON_CALIFICACION_ACIDA_MAYOR_25%
0,NaN,1,7434409712,NaN,10556,3034.0,NaN,0.287419,NaN,0.29,...,0.0,0.0,A,A,0.0,0.0,A,A,0.0,0.0
1,NaN,1,218820678,NaN,44431,5593.0,NaN,0.125881,NaN,0.13,...,0.0,0.0,A,A,0.0,0.0,A,A,0.0,0.0
2,NaN,1,643449365,NaN,7247,4897.0,NaN,0.675728,NaN,0.68,...,0.0,0.0,A,A,0.0,0.0,A,A,0.0,0.0


CRUZAR DATAFRAME DE DATOS INTERNOS CON CALIFICACION DE CENTRALES

In [43]:
# Asegurar mismo tipo de dato
data["IDETIFICACIÓN"] = data["IDETIFICACIÓN"].astype(str)
df_calif["NUMERO_ID"] = df_calif["NUMERO_ID"].astype(str)
df_calif = df_calif.drop_duplicates(subset="NUMERO_ID")
data_final = data.merge(
    df_calif[["NUMERO_ID", "PROVISION_CON_CALIFICACION_SUPER_MAYOR_25%"]], 
    left_on="IDETIFICACIÓN",
    right_on="NUMERO_ID",
    how="left"
)
print(data_final.shape)

(57860, 84)


In [44]:
data_final.columns

Index(['VIGENCIA', 'MES', 'DÍA', 'CTVO', 'TIPO IDENTIFICACIÓN',
       'IDETIFICACIÓN', 'CÓDIGO CONTABLE', 'MODIFICACIONES AL CRÉDITO',
       'CRÉDITO', 'CODCUE', 'FECHA DESEMBOLSO', 'FECHA VENCIMIENTO',
       'MOROSIDAD', 'TIPO CUOTA', 'CUOTAS PAGADAS', 'AMORTIZACIÓN',
       'MODALIDAD', 'TASA INTERÉS NOMINAL', 'TASA INTERÉS EFECTIVA',
       'VALOR PRÉSTAMO', 'VALOR CUOTA', 'SALDO CAPITAL', 'SALDO INTERESES',
       'OTROS SALDOS', 'GARANTÍA', 'FECHA AVALÚO', 'DETERIORO',
       'DETERIORO INTERESES', 'CONTINGENCIA', 'VALOR CUOTA EXTRA',
       'MESES CUOTA EXTRA', 'FECHA ÚLTIMO PAGO', 'CLASE GARANTÍA',
       'DESTINO CRÉDITO', 'CÓDIGO OFICINA', 'PERIODICIDAD AMORTIZACIÓN',
       'VALOR MORA', 'TIPO VIVIENDA', 'SEÑAL VIS ', 'TIPO O RANGO VIVIENDA',
       'SEÑAL SUBSIDIO', 'ENTIDAD REDESCUENTO', 'MARGEN REDESCUENTO',
       'DESEMBOLSO', 'MONEDA', 'FECHA MODIFICACIÓN',
       'CALIFICACIÓN ANTES MODIFICACIÓN', 'VALOR APORTES SOCIALES',
       'VALOR LÍNEA CRÉDITO', 'MODIFICACION

In [21]:
# Distribución de calificaciones (A, B, C, D, E)
calif_cols = ['PART_CALIF_A','PART_CALIF_B','PART_CALIF_C','PART_CALIF_D','PART_CALIF_E']

print('=== Estadísticas de participación por calificación ===')
print(df_calif[calif_cols].describe().round(3))

print('\n=== Clientes con calificación predominante ===')
for col in calif_cols:
    n = (df_calif[col] == 1.0).sum()
    pct = n / len(df_calif) * 100
    print(f'  {col[-1]}: {n:,} clientes ({pct:.1f}%)')

=== Estadísticas de participación por calificación ===
       PART_CALIF_A  PART_CALIF_B  PART_CALIF_C  PART_CALIF_D  PART_CALIF_E
count     54384.000     54384.000     54384.000     54384.000     54384.000
mean          0.594         0.005         0.004         0.008         0.034
std           0.306         0.040         0.039         0.055         0.129
min           0.000         0.000         0.000         0.000         0.000
25%           0.330         0.000         0.000         0.000         0.000
50%           0.600         0.000         0.000         0.000         0.000
75%           0.860         0.000         0.000         0.000         0.000
max           1.000         1.000         1.000         1.000         1.000

=== Clientes con calificación predominante ===
  A: 12,906 clientes (23.7%)
  B: 28 clientes (0.1%)
  C: 30 clientes (0.1%)
  D: 61 clientes (0.1%)
  E: 301 clientes (0.6%)


In [22]:
# Calificación de arrastre Superfinanciera
col_arrastre = 'CALIF_DE_ARRASTRE_SUPER_MAYOR_20%'
print('=== Calificación de arrastre Superfinanciera (>20%) ===')
print(df_calif[col_arrastre].value_counts().to_string())

print('\n=== Provisión con calificación Superfinanciera >20% ===')
prov = df_calif['PROVISION_CON_CALIFICACION_SUPER_MAYOR_20%']
print(f'  Total provisión: ${prov.sum():,.0f} miles COP')
print(f'  Clientes con provisión > 0: {(prov > 0).sum():,}')

=== Calificación de arrastre Superfinanciera (>20%) ===
CALIF_DE_ARRASTRE_SUPER_MAYOR_20%
A    44036
D     2537
E     2171
C      743
B      498

=== Provisión con calificación Superfinanciera >20% ===
  Total provisión: $25,328,083 miles COP
  Clientes con provisión > 0: 5,946


## 2. PEC Saldos + Calificación (273,745 obligaciones)

In [12]:
df_pec = pd.read_excel(
    os.path.join(RUTA, 'SALIDA PEC SALDOS_DIC_2025_centrales_.xlsx'),
    sheet_name='PEC SALDOS + CALIF'
)

print('Shape:', df_pec.shape)
df_pec.head(5)

Shape: (273745, 17)


,TIPO IDENTIFICACION,NUMERO IDENTIFICACION,NOMBRE,ESTADO DOCUMENTO,CALIDAD,SECTOR,PRODUCTO,NUMERO OBLIGACION,UTRAHUILCA,FECHA APERTURA,FECHA TERMINACION,CUPO/ VALOR INICIAL,SALDO,CUOTA O PAGO MINIMO,VALOR MORA,CALIFICACION,ALTURA MORA
0,1,8201278449,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,582,582.0,2019-10-11,2043-09-11,65600.0,57244.0,1607.0,0.0,NaN,Normal - Desde 0 hasta 29
1,1,978693734,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,1206,1206.0,2024-05-21,2044-06-05,67200.0,66313.0,670.0,0.0,NaN,Normal - Desde 0 hasta 29
2,1,394535961,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,7514,7514.0,2025-06-27,2045-07-02,144492.0,144540.0,1537.0,0.0,NaN,Normal - Desde 0 hasta 29
3,1,3203159666,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,4187,4187.0,2022-12-19,2038-06-19,251921.0,241239.0,3991.0,0.0,NaN,Normal - Desde 0 hasta 29
4,1,3569210360,NaN,VIGENTE,AVAL,SECTOR FINANCIERO,CARTERA VIVI,8017,8017.0,2015-02-16,2030-03-16,57400.0,23459.0,1075.0,0.0,NaN,Normal - Desde 0 hasta 29


In [13]:
# Distribución por producto
print('=== Obligaciones por producto ===')
print(df_pec['PRODUCTO'].value_counts().to_string())

print('\n=== Saldo por producto (miles COP) ===')
print(df_pec.groupby('PRODUCTO')['SALDO'].sum().sort_values(ascending=False).apply(lambda x: f'{x:,.0f}').to_string())

=== Obligaciones por producto ===
PRODUCTO
CARTERA SOLIDARIO       72377
SECTOR REAL COMERCIO    54251
SECTOR REAL SERVICIO    54238
TARJETA CREDITO         37343
CARTERA CONS            28122
CARTERA MICT            20300
CARTERA CIAL             3270
CARTERA VIVI             2238
CARTERA FIDUCIARIA         64
CARTERA LEASING            56
SECTOR ASEGURADOR          13
CARTERA OTRO                2

=== Saldo por producto (miles COP) ===
PRODUCTO
CARTERA SOLIDARIO       682,719,597
CARTERA CONS            537,901,888
CARTERA MICT            181,413,905
CARTERA VIVI            136,665,532
CARTERA CIAL            108,915,608
TARJETA CREDITO          83,368,886
SECTOR REAL COMERCIO     57,964,015
CARTERA LEASING           8,023,941
CARTERA FIDUCIARIA        1,432,202
SECTOR REAL SERVICIO      1,247,287
CARTERA OTRO                 58,393
SECTOR ASEGURADOR                 0


In [14]:
# Distribución por altura de mora
print('=== Distribución por altura de mora ===')
print(df_pec['ALTURA MORA'].value_counts().to_string())

print('\n=== Valor mora total ===')
mora_total = df_pec['VALOR MORA'].sum()
print(f'  ${mora_total:,.0f} miles COP')
print(f'  Obligaciones en mora > 0: {(df_pec["VALOR MORA"] > 0).sum():,}')

=== Distribución por altura de mora ===
ALTURA MORA
Normal - Desde 0 hasta 29              241010
Mora 360 - Desde 360 hasta 539           6001
Mora 730 - Mayor o Igual a 730 dias      5998
Mora 540 - Desde 540 hasta 729           4367
Mora 30  - Desde 30 hasta 59             2494
Mora 60  - Desde 60 hasta 89             2008
Mora 120 - Desde 120 hasta 149           1577
Mora 150 - Desde 150 hasta 179           1564
Mora 90  - Desde 90  hasta 119           1530
Mora 180 - Desde 180 hasta 209           1407
Mora 270 - Desde 270 hasta 299           1333
Mora 210 - Desde 210 hasta 239           1157
Mora 330 - Desde 330 hasta 359           1094
Mora 300 - Desde 300 hasta 329           1086
Mora 240 - Desde 240 hasta 269           1083

=== Valor mora total ===
  $127,763,121 miles COP
  Obligaciones en mora > 0: 33,257


In [15]:
# Distribución por sector
print('=== Saldo por sector ===')
print(df_pec.groupby('SECTOR')['SALDO'].agg(['sum','count']).sort_values('sum', ascending=False)
      .rename(columns={'sum':'Saldo total','count':'N obligaciones'})
      .apply(lambda col: col.apply(lambda x: f'{x:,.0f}')).to_string())

=== Saldo por sector ===
                       Saldo total N obligaciones
SECTOR                                           
SECTOR FINANCIERO      974,411,469         54,040
SECTOR SOLIDARIO       682,719,597         72,374
TARJETAS DE CREDITO     83,368,886         37,343
SECTOR REAL COMERCIO    57,964,015         54,230
SECTOR REAL SERVICIOS    1,247,287         54,238
SECTOR ASEGURADOR                0              0


## Modelo de calificación de cartera

In [17]:
df_pec.head()

,TIPO IDENTIFICACION,NUMERO IDENTIFICACION,NOMBRE,ESTADO DOCUMENTO,CALIDAD,SECTOR,PRODUCTO,NUMERO OBLIGACION,UTRAHUILCA,FECHA APERTURA,FECHA TERMINACION,CUPO/ VALOR INICIAL,SALDO,CUOTA O PAGO MINIMO,VALOR MORA,CALIFICACION,ALTURA MORA
0,1,8201278449,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,582,582.0,2019-10-11,2043-09-11,65600.0,57244.0,1607.0,0.0,NaN,Normal - Desde 0 hasta 29
1,1,978693734,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,1206,1206.0,2024-05-21,2044-06-05,67200.0,66313.0,670.0,0.0,NaN,Normal - Desde 0 hasta 29
2,1,394535961,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,7514,7514.0,2025-06-27,2045-07-02,144492.0,144540.0,1537.0,0.0,NaN,Normal - Desde 0 hasta 29
3,1,3203159666,NaN,VIGENTE,PRIN,SECTOR FINANCIERO,CARTERA VIVI,4187,4187.0,2022-12-19,2038-06-19,251921.0,241239.0,3991.0,0.0,NaN,Normal - Desde 0 hasta 29
4,1,3569210360,NaN,VIGENTE,AVAL,SECTOR FINANCIERO,CARTERA VIVI,8017,8017.0,2015-02-16,2030-03-16,57400.0,23459.0,1075.0,0.0,NaN,Normal - Desde 0 hasta 29


In [18]:
df_pec["CALIFICACION"].unique()

array([nan, 'A', 'B', 'E', 'AA', 'C', 'K', 'D', 'BB', 'CC',
       'No Calificado'], dtype=object)